In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types

In [2]:
from pyspark.sql import functions as F

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

25/03/07 20:28:34 WARN Utils: Your hostname, codespaces-029a01 resolves to a loopback address: 127.0.0.1; using 10.0.0.246 instead (on interface eth0)
25/03/07 20:28:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/07 20:28:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
spark.version

'3.5.5'

In [5]:
df_yellow = spark.read.parquet('data/raw/yellow_tripdata_2024-10.parquet')

df_yellow.columns

In [6]:
df_yellow \
    .repartition(4) \
    .write.parquet('data/pq/homework')

In [6]:
df_yellow = df_yellow \
    .withColumnRenamed("tpep_pickup_datetime", "pickup_datetime") \
    .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")

In [20]:
# count trips per date
df_yellow \
    .withColumn('pickup_date', F.to_date('pickup_datetime')) \
    .filter('trip_distance > 0') \
    .groupBy('pickup_date') \
    .count() \
    .orderBy('pickup_date').show()

[Stage 19:=============================>                            (1 + 1) / 2]

+-----------+------+
|pickup_date| count|
+-----------+------+
| 2009-01-01|     1|
| 2024-09-30|    12|
| 2024-10-01|116756|
| 2024-10-02|111604|
| 2024-10-03|106308|
| 2024-10-04|109018|
| 2024-10-05|120772|
| 2024-10-06|100490|
| 2024-10-07| 99876|
| 2024-10-08|119167|
| 2024-10-09|126916|
| 2024-10-10|140055|
| 2024-10-11|124665|
| 2024-10-12|128298|
| 2024-10-13|108232|
| 2024-10-14|100080|
| 2024-10-15|126106|
| 2024-10-16|131798|
| 2024-10-17|133411|
| 2024-10-18|129429|
+-----------+------+
only showing top 20 rows



In [24]:
@F.udf(returnType=types.StringType())
def trip_duration(time_end, time_start):
    diff = (time_end-time_start).total_seconds()
    out_string = ""
    if diff > 3600:
        hours = int(diff / 3600)
        diff = diff % 3600
        out_string += f"{hours}h"
    if diff > 60:
        mins =int(diff / 60)
        diff = diff % 60
        out_string += f"{mins}m"
    if diff > 0:
        secs = diff
        out_string += f"{secs}s"
    return out_string
        

In [30]:
# find the longest trip in hours
df_yellow \
    .where("date(dropoff_datetime) != date(pickup_datetime)") \
    .withColumn('trip_duration', trip_duration('dropoff_datetime','pickup_datetime')) \
    .withColumn('trip_time', F.date_diff('dropoff_datetime','pickup_datetime')) \
    .select(['trip_duration', 'dropoff_datetime','pickup_datetime']) \
    .orderBy(F.desc('trip_time')) \
    .limit(10).show()


[Stage 24:>                                                         (0 + 2) / 2]

+-------------+-------------------+-------------------+
|trip_duration|   dropoff_datetime|    pickup_datetime|
+-------------+-------------------+-------------------+
|  162h37m4.0s|2024-10-23 07:40:53|2024-10-16 13:03:49|
| 137h45m38.0s|2024-10-28 09:46:33|2024-10-22 16:00:55|
| 143h19m30.0s|2024-10-09 18:06:55|2024-10-03 18:47:25|
|  114h50m5.0s|2024-10-23 04:43:37|2024-10-18 09:53:32|
|  89h26m46.0s|2024-10-24 06:57:38|2024-10-20 13:30:52|
|  67h34m24.0s|2024-10-15 15:07:15|2024-10-12 19:32:51|
|  89h53m54.0s|2024-10-24 18:30:18|2024-10-21 00:36:24|
|        66h4m|2024-10-20 12:02:18|2024-10-17 17:58:18|
|  70h17m57.0s|2024-10-25 14:22:49|2024-10-22 16:04:52|
|  42h18m32.0s|2024-10-22 13:17:00|2024-10-20 18:58:28|
+-------------+-------------------+-------------------+



Finding the least frequent pickup zone

In [7]:
df_zones = spark.read \
    .option("header", "true") \
    .csv('../../Week4-dbt/seeds/taxi_zones_lookup.csv')

In [8]:
df_zones.head(5)

[Row(locationid='1', borough='EWR', zone='Newark Airport', service_zone='EWR'),
 Row(locationid='2', borough='Queens', zone='Jamaica Bay', service_zone='Boro Zone'),
 Row(locationid='3', borough='Bronx', zone='Allerton/Pelham Gardens', service_zone='Boro Zone'),
 Row(locationid='4', borough='Manhattan', zone='Alphabet City', service_zone='Yellow Zone'),
 Row(locationid='5', borough='Staten Island', zone='Arden Heights', service_zone='Boro Zone')]

In [9]:
df_expanded = df_yellow.join(df_zones, df_zones.locationid == df_yellow.PULocationID)

In [17]:
df_expanded \
    .select(['zone']) \
    .groupBy('zone') \
    .count() \
    .orderBy(F.asc('count')) \
    .limit(10).show()

+--------------------+-----+
|                zone|count|
+--------------------+-----+
|Governor's Island...|    1|
|       Rikers Island|    2|
|       Arden Heights|    2|
|         Jamaica Bay|    3|
| Green-Wood Cemetery|    3|
|Charleston/Totten...|    4|
|   Rossville/Woodrow|    4|
|       West Brighton|    4|
|       Port Richmond|    4|
|Eltingville/Annad...|    4|
+--------------------+-----+

